In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

In [2]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),          # data augmentation
    transforms.RandomCrop(32, padding=4),       # data augmentation
    transforms.ColorJitter(brightness=0.2,
                           contrast=0.2,
                           saturation=0.2),     # data augmentation
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])
 
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])


In [3]:


trainset=CIFAR10(root="./data",train=True,download=True,transform=train_transform)
testset=CIFAR10(root="./data",train=False,download=True,transform=train_transform)


from torch.utils.data import random_split

train_size = int(0.8 * len(trainset))
val_size = len(trainset) - train_size

trainset, valset = random_split(trainset, [train_size, val_size])

Files already downloaded and verified
Files already downloaded and verified


In [ ]:

trainloader = DataLoader(trainset, batch_size=64, shuffle=True,
                         num_workers=4, pin_memory=True)  
valloader   = DataLoader(valset,   batch_size=64, shuffle=False,
                         num_workers=4, pin_memory=True)
testloader  = DataLoader(testset,  batch_size=64, shuffle=False,
                         num_workers=4, pin_memory=True)

###build the CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
 
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),                 # dropout – cuts overfitting
 
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),                 # dropout
 
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),                 # dropout
        )
 
        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),
            nn.Dropout(0.5),                    # dropout in FC layer
            nn.Linear(256, 10)
        )
 
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)              # flatten
        x = self.fc_layers(x)
        return x

In [6]:
model=CNN()

In [7]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

In [8]:
EPOCHS        = 50
PATIENCE      = 5           # stop if val loss doesn't improve for 5 epochs
best_val_loss = float("inf")
epochs_no_imp = 0
best_weights  = None
 

for epoch in range(EPOCHS):

    # -------- TRAIN --------
    model.train()
    train_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # -------- VALIDATION --------
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in valloader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()


    avg_train = train_loss / len(trainloader)
    avg_val   = val_loss   / len(valloader)
    val_accuracy = 100 * correct / total

    scheduler.step(avg_val)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss/len(trainloader):.4f}")
    print(f"Val Loss: {val_loss/len(valloader):.4f}")
    print(f"Val Accuracy: {val_accuracy:.2f}%")

     #EARLY STOPPING 
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        epochs_no_imp = 0
        best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_imp += 1
        if epochs_no_imp >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch+1}.")
            break
 
# Restore best weights
model.load_state_dict(best_weights)


Epoch 1
Train Loss: 1.6656
Val Loss: 1.4168
Val Accuracy: 48.34%
Epoch 2
Train Loss: 1.2938
Val Loss: 1.2194
Val Accuracy: 56.45%
Epoch 3
Train Loss: 1.1096
Val Loss: 1.0742
Val Accuracy: 62.07%
Epoch 4
Train Loss: 0.9849
Val Loss: 0.9839
Val Accuracy: 65.84%
Epoch 5
Train Loss: 0.9046
Val Loss: 0.9177
Val Accuracy: 67.55%
Epoch 6
Train Loss: 0.8422
Val Loss: 0.8880
Val Accuracy: 68.62%
Epoch 7
Train Loss: 0.7958
Val Loss: 0.8340
Val Accuracy: 70.23%
Epoch 8
Train Loss: 0.7629
Val Loss: 0.8144
Val Accuracy: 71.25%
Epoch 9
Train Loss: 0.7273
Val Loss: 0.7828
Val Accuracy: 72.66%
Epoch 10
Train Loss: 0.6935
Val Loss: 0.7604
Val Accuracy: 73.92%
Epoch 11
Train Loss: 0.6790
Val Loss: 0.7580
Val Accuracy: 73.52%
Epoch 12
Train Loss: 0.6518
Val Loss: 0.7295
Val Accuracy: 74.48%
Epoch 13
Train Loss: 0.6370
Val Loss: 0.7229
Val Accuracy: 74.72%
Epoch 14
Train Loss: 0.6208
Val Loss: 0.7118
Val Accuracy: 75.52%
Epoch 15
Train Loss: 0.6066
Val Loss: 0.7171
Val Accuracy: 75.07%
Epoch 16
Train Loss

<All keys matched successfully>

In [12]:
classes = ["airplane","automobile","bird","cat","deer",
           "dog","frog","horse","ship","truck"]
 
model.eval()
all_preds  = []
all_labels = []
 
with torch.no_grad():
    for images, labels in testloader:
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
 
# Overall accuracy
accuracy = 100 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"\nTest Accuracy: {accuracy:.2f}%")
 
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(cm)
 
# Precision, Recall, F1
report = classification_report(all_labels, all_preds, target_names=classes)
print("\nClassification Report (Precision / Recall / F1):")
print(report)


Test Accuracy: 81.30%

Confusion Matrix:
[[816  14  37  23  12   2   6   9  51  30]
 [ 13 896   4   4   0   3   6   0  18  56]
 [ 39   3 738  40  52  48  51  15   7   7]
 [ 19   6  44 625  45 144  61  26  13  17]
 [ 15   4  50  27 795  32  37  31   7   2]
 [  8   2  23 133  29 757  13  29   5   1]
 [  6   3  31  41  24   7 877   3   7   1]
 [  9   2  14  40  30  43   1 850   3   8]
 [ 33  10  13   6   3   3   4   4 910  14]
 [ 24  54   5   9   2   3   5   8  24 866]]

Classification Report (Precision / Recall / F1):
              precision    recall  f1-score   support

    airplane       0.83      0.82      0.82      1000
  automobile       0.90      0.90      0.90      1000
        bird       0.77      0.74      0.75      1000
         cat       0.66      0.62      0.64      1000
        deer       0.80      0.80      0.80      1000
         dog       0.73      0.76      0.74      1000
        frog       0.83      0.88      0.85      1000
       horse       0.87      0.85      0.86 

TypeError: 'int' object is not iterable